### The Henon-Heiles Hamiltonian

$$
H = \frac{1}{2}(p_1^2 + p_2^2) + \frac{1}{2}(q_1^2 + q_2^2) + \alpha q_1^2 q_2 - \frac{\beta}{3} q_2^3
$$

### How to run

- In order to obtain the PySR regressed equations we need to provide data of a trajectory predicted by the ASRNN.
- This is provided in the code by specififying a file path, we can use the example data in the repository or additional data in the Supplementary Material, or choose to generate new data using the repository and provide its file path in the next cell.
- Unlike for PySINDy, PySR is constructed to take in time shuffled values of $\mathbf{q}, \mathbf{p}$ and regress the Kinetic and Potential energies predicted by the ASRNN, hence data of multiple trajectories can be fed at once.
- Note that PySR requires building a Python-Julia bridge which can often run differently on different machines, this code runs on google colab. Ocassionally due to troubles with the bridge depending on versions of numpy and other libraries, the code sometimes throws back a unidecode error, this can largely be ignored and the code will continue running in spite of it. If it persists the kernel may need a restart.

In [1]:
#### Install PySR #####
!pip install -U pysr

In [2]:
##### Import PySR and others (this may take some time) #######
import pysr
import sympy
import numpy as np
from matplotlib import pyplot as plt
from pysr import PySRRegressor

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [3]:
######### Input file path to trajectory data ############

file_path = 'predictions/hamiltonian_mlp_predictions.npz'

In [4]:
data = np.load(file_path)
shape = data['qs_pred'].shape

q = data['qs_pred'].reshape((shape[0]*shape[1], 2), order='F')
p = data['ps_pred'].reshape((shape[0]*shape[1], 2), order='F')
K = data['Ks_pred'].reshape(shape[0]*shape[1], order='F')
V = data['Vs_pred'].reshape(shape[0]*shape[1], order='F')

def shuffle(a, b, c, d): #function to shuffle data to destroy temporal structure
    l = a.shape[0]
    randomize = np.arange(l)
    np.random.shuffle(randomize)
    a_shuffled = a[randomize]
    b_shuffled = b[randomize]
    c_shuffled = c[randomize]
    d_shuffled = d[randomize]

    return a_shuffled, b_shuffled, c_shuffled, d_shuffled

q_shuffled, p_shuffled, K_shuffled, V_shuffled = shuffle(q, p, K, V)

In [8]:
print(data['params'])

[[0.5 0.7]
 [0.5 0.7]
 [0.5 0.7]
 [0.5 0.7]
 [0.5 0.7]
 [0.5 0.7]
 [0.5 0.7]
 [0.5 0.7]
 [0.5 0.7]
 [0.5 0.7]]


In [5]:
default_pysr_params = dict(
    populations=30,
    model_selection="best",
)

### Kinetic Energy regression

In [8]:
model = PySRRegressor(
    niterations=30,
    binary_operators=["+", "-", "*", "myoperator(x, y) = x * y"],
    unary_operators=['square', 'cube'],
    extra_sympy_mappings={"myoperator" : lambda x, y: x * y},
    **default_pysr_params
)

model.fit(p_shuffled[:1000], K_shuffled[:1000]) #Random sample of 1000 points

/usr/local/lib/python3.11/dist-packages/pysr/sr.py:2766: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 7.760e+03
Progress: 82 / 900 total iterations (9.111%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           1.616e-03  1.594e+01  y = -4.0277
5           1.615e-03  1.328e-04  y = -4.0277 - cube(cube(x₀))
6           7.471e-04  7.710e-01  y = square(myoperator(x₁, x₁)) + -4.0388
7           4.730e-04  4.571e-01  y = -4.0587 - (x₁ * (x₁ * -0.43437))
10          4.730e-04  -0.000e+00  y = ((myoperator(square(x₁), 0.43431) + -2.6993) - 0.9417...
                                       7) + -0.41769
15          4.730e-04  2.384e-08  y = (((-1.398 - myoperator(x₁, myoperator(-0.4343, x₁))) -...
                                       (x₀ + 1.7769)) + x₀) - 0.88378
19          4.728e-04  1.172e-04  y = (((-2.079 - myoperator(x₁, (myoperator(1.9326, x₁) * -...


[ Info: Final population:
[ Info: Results saved to:


PySRRegressor.equations_ = [
	    pick      score                                           equation  \
	0          0.000000                                          -4.027669   
	1          0.192895                      square(square(x1)) - 4.038844   
	2          0.457071              (square(x1) * 0.43431005) + -4.058723   
	3          0.265995  square(myoperator(-1.3371251, square(x0) + squ...   
	4         12.865794  (square(x1) * 0.49908113) + (myoperator(square...   
	5          0.000335  (myoperator(x0, myoperator(x0, 0.4991653)) + (...   
	6   >>>>   0.093154  (square(x0 * -0.7064885) - ((-6.337158e-5 - x1...   
	7          0.084827  (-4.0860324 - (x1 * myoperator(x1, (-0.0002540...   
	8          0.006435  square(x0 * -0.70648205) + (-4.0860324 - myope...   
	9          0.001096  square(x0 * -0.7064826) + (-4.086033 - ((myope...   
	10         0.003725  (square(x0 * -0.7064702) + -4.086032) - ((((cu...   
	11         0.015618  square(myoperator(x0, -0.7064776)) + (-4.08603...   
	12         0.015543  (-4.0860333 - (x1 * (myoperator((((x0 + -0.357...   
	
	            loss  complexity  
	0   1.616204e-03           1  
	1   7.471411e-04           5  
	2   4.730413e-04           6  
	3   1.632367e-04          10  
	4   4.219631e-10          11  
	5   4.218215e-10          12  
	6   3.501193e-10          14  
	7   2.954856e-10          16  
	8   2.917068e-10          18  
	9   2.910679e-10          20  
	10  2.899858e-10          21  
	11  2.810679e-10          23  
	12  2.641254e-10          27  
]

  - outputs/20250129_155914_4Wcvrk/hall_of_fame.csv


### Potential Energy regression

In [6]:
model2 = PySRRegressor(
    niterations=50,
    binary_operators=["+", "-", "*", 'myoperator(x, y) = x * y'],
    unary_operators=['cube', 'square'],
    extra_sympy_mappings={"myoperator" : lambda x, y: x * y},
    **default_pysr_params
)

model2.fit(q_shuffled[:1000], V_shuffled[:1000])  #Random sample of 1000 points

/usr/local/lib/python3.11/dist-packages/pysr/sr.py:2766: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


Compiling Julia backend...


[ Info: Started!



Expressions evaluated per second: 2.540e+04
Progress: 102 / 1500 total iterations (6.800%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           1.763e-03  1.594e+01  y = -4.3224
5           1.725e-03  5.440e-03  y = -4.3232 - (x₁ * -0.021736)
6           8.213e-04  7.423e-01  y = square(x₁ * x₁) + -4.3383
13          7.637e-04  1.038e-02  y = (square(square(x₁ + myoperator(square(square(x₀)), -3....
                                      2655))) - 0.085024) + -4.2562
17          4.754e-04  1.185e-01  y = ((square(square(x₁)) - 0.085024) + -4.2562) + myoperat...
                                      or(square(x₁) + 0.058366, square(x₀ * -1.5342))
20          4.130e-04  4.688e-02  y = ((myoperator(square(square(x₀) - cube(x₁)), (x₁ - 0.59...
                                      03) * -2.

[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           1.763e-03  1.594e+01  y = -4.3224
5           8.213e-04  1.910e-01  y = square(square(x₁)) + -4.3383
6           5.802e-04  3.475e-01  y = myoperator(square(x₁), 0.3591) + -4.3518
7           5.792e-04  1.723e-03  y = cube((square(x₁) * 0.045009) + -1.6326)
8           5.181e-04  1.113e-01  y = (square(x₁ + -0.038305) * 0.39388) + -4.3541
9           1.055e-04  1.591e+00  y = ((square(x₀) + square(x₁)) * 0.42239) + -4.3777
11          7.655e-05  1.606e-01  y = ((square(x₁ + -0.022548) + square(x₀)) * 0.43852) + -4...
                                      .3794
13          7.655e-05  2.742e-06  y = (myoperator(x₁, -0.019778) + myoperator(0.4385, square...
                                      (x₀) + square(x₁))) + -4.3791
14          4.025e-05  6.429e-01  y = cube(x₁ * -0.53451) + (-4.382 + ((square(x₀) + square(...
                  

PySRRegressor.equations_ = [
	    pick     score                                           equation  \
	0         0.000000                                           -4.32244   
	1         0.191015                     square(square(x1)) + -4.338276   
	2         0.347524    myoperator(square(x1), 0.35910094) + -4.3517556   
	3         0.001723      cube((square(x1) * 0.045008805) + -1.6325811)   
	4         0.111348  (square(x1 + -0.03830504) * 0.39387745) + -4.3...   
	5         1.591138  ((square(x0) + square(x1)) * 0.42238745) + -4....   
	6         0.160569  ((square(x1 + -0.022547837) + square(x0)) * 0....   
	7         0.000003  (myoperator(x1, -0.019777576) + myoperator(0.4...   
	8         0.642947  cube(x1 * -0.5345117) + (-4.3820367 + ((square...   
	9         0.004984  myoperator(square(x1), (x1 * -0.15284209) + 0....   
	10        0.001029  cube((square(x0) * 0.056744833) + ((square(x1)...   
	11        0.084847  (square(x0) * 0.4681938) + (myoperator(square(...   
	12  >>>>  7.213890  (myoperator(square(x1), myoperator(-0.23212013...   
	
	            loss  complexity  
	0   1.763252e-03           1  
	1   8.212728e-04           5  
	2   5.801757e-04           6  
	3   5.791768e-04           7  
	4   5.181474e-04           8  
	5   1.055434e-04           9  
	6   7.655301e-05          11  
	7   7.655258e-05          13  
	8   4.024681e-05          14  
	9   4.004672e-05          15  
	10  4.000555e-05          16  
	11  3.376157e-05          18  
	12  2.485822e-08          19  
]

  - outputs/20250129_160648_rAQUhb/hall_of_fame.csv


# Output

Note that sometimes spurious terms of very small coefficients (atleast an order of magnitude smaller than the rest) may be recovered. Also both the kinetic and potential energies can always be offset by a constant and that is expected in the outputs

In [9]:
sympy.expand(model.sympy())

0.49912600063225*x0**2 + 0.49906906*x1**2 + 3.16267948613148e-5*x1 - 4.086034

In [7]:
sympy.expand(model2.sympy())

0.4973462*x0**2*x1 + 0.50034857*x0**2 - 0.23212013*x1**3 + 0.5001039*x1**2 - 4.3841753